# Days 13–15: FNO Study, Implementation, and Training

## Day 13 — FNO Architecture Review

### What a Fourier Neural Operator is

A standard MLP applies learnable linear transforms in **physical space** — every input feature interacts with every hidden unit through a weight matrix. An FNO applies its core transform in **frequency space** instead.

For a 1D input signal with N points:

1. **FFT** — convert the spatial signal to Fourier coefficients
2. **Truncate** — keep only the lowest K frequency modes (discard high frequencies)
3. **Linear transform** — apply a learnable complex-valued weight matrix R to those K modes
4. **IFFT** — convert back to spatial domain

This is the **SpectralConv1d** layer. It captures global structure efficiently because smooth physical fields (like pressure distributions) have most of their energy in low-frequency modes.

### The residual bypass (W)

Each FNO block adds a **pointwise linear bypass** (1×1 convolution) to the spectral output:

```
output = GELU( SpectralConv(x) + W(x) )
```

W handles local/high-frequency features that the truncated spectral path misses.

### Full FNO1d architecture for this project

```
Input: (batch, 201, 4)
  — 201 spatial positions (198 geometry + 3 scalar conditions)
  — 4 channels: [geometry_y, alpha, Reynolds, Ncrit] at each position

Lift P:       (batch, 201, 4)  →  (batch, 201, width)
FNO Block ×4: (batch, width, 201) → (batch, width, 201)  [spectral + bypass]
Flatten:      (batch, width, 201) → (batch, width*201)
Project Q:    Linear → GELU → Linear → (batch, 100)

Output: (batch, 100) — [Cl, Cd, Cp×98]
```

### Why FNO1d and not a fancier variant

- **U-FNO, FFNO** — designed for 2D/3D domains with skip connections between resolutions. Overkill for a 1D signal-to-signal mapping.
- **Geo-FNO** — for unstructured meshes. Our geometry is already a fixed-length 1D array.
- **FNO1d (Li et al. 2021)** — exact match: 1D input signal → 1D output signal, conditioned on scalar parameters. This is what we implement.

### Hyperparameters to tune (Day 17)

| Parameter | Role | Day 15 default |
|-----------|------|----------------|
| `modes` K | Number of Fourier modes retained | 16 |
| `width`   | Channel dimension throughout FNO | 64 |
| `n_layers` | Number of FNO blocks | 4 |

Increasing K captures finer spatial structure but costs memory. Increasing width increases capacity but training time. Start with the defaults and tune in Day 17.

---

## Day 14 — Implementation and Grid Representation

Implementation lives in `train_fno.py`. Key design decisions documented below.

### Grid representation

The FNO treats its 201-point input as a 1D spatial grid. Each grid point has 4 channels:
- Channel 0: geometry_y value at that position (the airfoil surface height)
- Channels 1–3: alpha, Reynolds, Ncrit — scalar conditions **broadcast to every position**

Broadcasting scalars to every position is standard practice for conditioning FNOs on global parameters. It gives the spectral convolution access to the operating condition at every frequency mode, not just as a global offset.

The 3 scalar positions (indices 198–200) in the geometry channel are zero-padded since there's no geometric meaning there — they still receive the scalar condition channels.

### Output

After the FNO blocks, the spatial+channel tensor is flattened and projected to 100 outputs via two linear layers. This is a global prediction head — the FNO processes the full geometry signal and produces scalar/vector outputs, not a field-to-field mapping. Identical output layout to the MLP and PINN for direct comparison.

---

## Day 15 — Training

In [ ]:
# ── Setup (run on fresh Colab session only) ───────────────────────────────
# !wget https://nasa-public-data.s3.amazonaws.com/plot3d_utilities/dataset-processed.zip
# !unzip -q dataset-processed.zip
# !python data/preprocess_airfoil_dataset.py \
#     --input_dir datasets/standard \
#     --output_dir processed_output \
#     --scaler_name standard

In [ ]:
import sys
sys.path.insert(0, '..')        # repo root — for train_fno.py
sys.path.insert(0, '../data')   # data/ — for load_dataset.py

import json
import numpy as np
import torch
import matplotlib.pyplot as plt

# Canonical data loader shared across all notebooks in this repo
from load_dataset import load_split

# FNO-specific helpers and model class from the training script
import train_fno
from train_fno import FNO1d, make_X_fno, make_y

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

In [ ]:
# ── Quick architecture sanity check ──────────────────────────────────────
model = FNO1d(in_channels=4, out_dim=100, n_points=201, modes=16, width=64, n_layers=4)
x_dummy = torch.randn(8, 201, 4)   # (batch=8, n_points=201, channels=4)
out = model(x_dummy)
print(f"Input:  {x_dummy.shape}")
print(f"Output: {out.shape}   (expected: torch.Size([8, 100]))")
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {n_params:,}")

In [ ]:
# ── Verify data shapes ────────────────────────────────────────────────────
DATA_DIR = "processed_output/standard/main"
train_split = load_split(f"{DATA_DIR}/train.npz")
val_split   = load_split(f"{DATA_DIR}/val.npz")

X_train_fno = make_X_fno(train_split)   # (N, 201, 4)
y_train     = make_y(train_split)        # (N, 100)
X_val_fno   = make_X_fno(val_split)
y_val       = make_y(val_split)

print(f"X_train: {X_train_fno.shape}  y_train: {y_train.shape}")
print(f"X_val:   {X_val_fno.shape}    y_val:   {y_val.shape}")
print()
print("Channel layout at position 0, sample 0:")
print(f"  ch0 (geometry_y):  {X_train_fno[0, 0, 0]:.4f}")
print(f"  ch1 (alpha):       {X_train_fno[0, 0, 1]:.4f}")
print(f"  ch2 (reynolds):    {X_train_fno[0, 0, 2]:.4f}")
print(f"  ch3 (ncrit):       {X_train_fno[0, 0, 3]:.4f}")
print()
print("Confirming scalars are broadcast (same at every position):")
print(f"  alpha same across positions: {np.all(X_train_fno[0, :, 1] == X_train_fno[0, 0, 1])}")

In [ ]:
# ── Run training via %run ─────────────────────────────────────────────────
# This calls train_fno.py as a script, equivalent to running it from the
# terminal. Saves artifacts/fno_v1.pt and artifacts/fno_v1_metrics.json.
#
# Adjust --epochs and --modes as needed.
# Recommended starting point: modes=16, width=64, 20 epochs.
# If val loss is still decreasing at epoch 20, extend to 40.

%run ../train_fno.py \
    --data_dir processed_output/standard/main \
    --output_dir artifacts \
    --epochs 20 \
    --modes 16 \
    --width 64 \
    --n_layers 4 \
    --lr 1e-3 \
    --batch_size 1024

In [ ]:
# ── Load results and print comparison table ───────────────────────────────
with open("artifacts/fno_v1_metrics.json") as f:
    results = json.load(f)

fno = results["val_metrics"]
mlp = results["mlp_baseline"]

print("=" * 65)
print(f"FNO1d best checkpoint: epoch {results['best_epoch']}  "
      f"({results['n_params']:,} params)")
print(f"Hyperparams: modes={results['hyperparams']['modes']}  "
      f"width={results['hyperparams']['width']}  "
      f"n_layers={results['hyperparams']['n_layers']}")
print("=" * 65)
print(f"{'':6} {'MAE':>8} {'RMSE':>8} {'R²':>8}   "
      f"{'MLP MAE':>8} {'MLP R²':>8}")
print("-" * 65)
for target in ["cl", "cd", "cp"]:
    print(f"{target.upper():6} "
          f"{fno[target]['mae']:8.4f} "
          f"{fno[target]['rmse']:8.4f} "
          f"{fno[target]['r2']:8.4f}   "
          f"{mlp[target]['mae']:8.4f} "
          f"{mlp[target]['r2']:8.4f}")
print("=" * 65)

In [ ]:
# ── Loss curves ───────────────────────────────────────────────────────────
history = results["history"]
epochs  = [h["epoch"]      for h in history]
tr_loss = [h["train_loss"] for h in history]
vl_loss = [h["val_loss"]   for h in history]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(epochs, tr_loss, label="Train (weighted MSE)", linewidth=2)
ax.plot(epochs, vl_loss, label="Val (weighted MSE)",   linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("FNO1d Training — Loss Curves")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("artifacts/fno_loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: artifacts/fno_loss_curves.png")

In [ ]:
# ── Cp visual comparison: MLP vs FNO vs ground truth ─────────────────────
model = FNO1d(
    in_channels=4, out_dim=100, n_points=201,
    modes=results["hyperparams"]["modes"],
    width=results["hyperparams"]["width"],
    n_layers=results["hyperparams"]["n_layers"]
).to(DEVICE)

ckpt = torch.load("artifacts/fno_v1.pt", map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"])
model.eval()

X_val_t = torch.tensor(X_val_fno, dtype=torch.float32).to(DEVICE)
with torch.no_grad():
    fno_pred = model(X_val_t).cpu().numpy()

sample_indices = [0, 500, 1000]
x_surface = np.linspace(0, 1, 98)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for i, idx in enumerate(sample_indices):
    ax = axes[i]
    ax.plot(x_surface, y_val[idx, 2:],      label="Ground Truth", color="steelblue", linewidth=2)
    ax.plot(x_surface, fno_pred[idx, 2:],   label="FNO1d",        color="tomato",    linewidth=2, linestyle="--")
    ax.invert_yaxis()
    ax.set_xlabel("Chord Position (x/c)")
    ax.set_ylabel("Cp")
    ax.set_title(f"Val Sample {idx}")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle("Cp: Ground Truth vs FNO1d", fontsize=13)
plt.tight_layout()
plt.savefig("artifacts/fno_cp_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: artifacts/fno_cp_comparison.png")

In [ ]:
# ── Convergence check ─────────────────────────────────────────────────────
# If val loss is still decreasing at the last epoch, extend training.

last_5_val = [h["val_loss"] for h in history[-5:]]
still_improving = last_5_val[-1] < last_5_val[0]

print("Last 5 val losses:", [f"{v:.6f}" for v in last_5_val])
if still_improving:
    print("⚠ Val loss still decreasing — consider extending epochs in Day 17.")
    print("  Re-run with --epochs 40 to check if further improvement is possible.")
else:
    print("✓ Val loss has plateaued — current epoch count is sufficient.")

print("\nDay 15 complete.")
print("Artifacts:")
print("  artifacts/fno_v1.pt")
print("  artifacts/fno_v1_metrics.json")
print("  artifacts/fno_loss_curves.png")
print("  artifacts/fno_cp_comparison.png")
print("\nNext: Day 16 — Coefficient prediction head, Day 17 — FNO hyperparameter tuning.")